### Conversation-With-History

In [1]:
!pip --version

pip 26.2.1 from d:\hanhwa0902\ex0918\.0918venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !pip install langchain_teddynote

In [4]:
from langchain_teddynote import logging

logging.langsmith("test0918")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0918


In [5]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 질의응답 챗봇입니다. 주어진 질문에 대한 답변을 제공해주세요.",),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "#Question:\n{question}"),
    ]
)

llm = ChatOpenAI(model="gpt-5-mini")

chain = prompt | llm | StrOutputParser()

In [11]:
store = {}

def get_session_history(session_ids):
    print(f"[대화 세션ID]: {session_ids}")
    if session_ids not in store:
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]

In [12]:
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [13]:
chain_with_history.invoke(
    {"question": "나의 이름은 로지입니다."},
    config={"configurable": {"session_id": "abc123"}},
)

[대화 세션ID]: abc123


"안녕하세요, 로지님. 반갑습니다! 무엇을 도와드릴까요? 계속해서 '로지님'이라고 불러드려도 될까요?"

In [14]:
chain_with_history.invoke(
    {"question": "내 이름이 뭐라고?"},
    config={"configurable": {"session_id": "abc123"}},
)

[대화 세션ID]: abc123


'로지님이에요.'

In [15]:
chain_with_history.invoke(
    {"question": "내 이름이 뭐라고?"},
    config={"configurable": {"session_id": "qwer123"}},
)

[대화 세션ID]: qwer123


'모르겠어요 — 당신이 아직 이름을 알려주지 않으셨어요. 원하시면 지금 알려주시겠어요? 알려주시면 이 대화에서는 그 이름으로 부를게요. (세션이 끝나면 제가 기억하진 못합니다.)'